In [ ]:

import os
import re
import ast
import json
import zipfile
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(r"C:\Users\hp\Analysis in Global Supply Chain Operations")
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "APL_Logistics.xlsx"
CLEAN_FILE = PROJECT_ROOT / "data" / "processed" / "cleaned_logistics_data.csv"
GAP_FILE = PROJECT_ROOT / "data" / "processed" / "delivery_gap_analysis.csv"
APP_FILE = PROJECT_ROOT / "app.py"

REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
ARCHIVE_DIR = PROJECT_ROOT / "outputs" / "final_archives"

for folder in [
    REPORT_DIR, TABLE_DIR, ARCHIVE_DIR,
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "outputs" / "forecasts"
]:
    folder.mkdir(parents=True, exist_ok=True)

results = []

def heading(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def record(name, status, details=""):
    results.append({
        "Validation": name,
        "Status": status,
        "Details": str(details)
    })
    print(f"{status:8} | {name} | {details}")

def normalize(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def find_exact_column(df, candidates):
    if df is None:
        return None
    lookup = {normalize(c): c for c in df.columns}
    for candidate in candidates:
        key = normalize(candidate)
        if key in lookup:
            return lookup[key]
    return None

def read_csv(path):
    if not path.exists():
        return None
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception:
        try:
            return pd.read_csv(path, low_memory=False, encoding="latin1")
        except Exception:
            return None

def find_files(roots, patterns):
    found = []
    for root in roots:
        if root.exists():
            for pattern in patterns:
                found.extend(root.rglob(pattern))
    return sorted(set(p for p in found if p.is_file()))

def valid_document(path):
    try:
        if path.suffix.lower() == ".pdf":
            with open(path, "rb") as f:
                header = f.read(8)
                f.seek(max(0, path.stat().st_size - 2048))
                tail = f.read()
            return header.startswith(b"%PDF") and b"%%EOF" in tail
        if path.suffix.lower() == ".docx":
            with zipfile.ZipFile(path) as z:
                return "word/document.xml" in z.namelist()
        return path.stat().st_size > 0
    except Exception:
        return False

def extract_docx_text(path):
    try:
        with zipfile.ZipFile(path) as z:
            xml = z.read("word/document.xml").decode("utf-8", errors="ignore")
        xml = re.sub(r"<[^>]+>", " ", xml)
        return re.sub(r"\s+", " ", xml).lower()
    except Exception:
        return ""

def write_text(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")

heading("PROJECT DIRECTORY AND DATASET LOADING")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_ROOT}")

raw_df = None
clean_df = read_csv(CLEAN_FILE)
gap_df = read_csv(GAP_FILE)

try:
    if RAW_FILE.exists():
        raw_df = pd.read_excel(RAW_FILE)
        record("Raw dataset verification", "PASS", f"Shape={raw_df.shape}")
    else:
        record("Raw dataset verification", "FAIL", f"Missing: {RAW_FILE}")
except Exception as error:
    record("Raw dataset verification", "FAIL", error)

record(
    "Cleaned dataset verification",
    "PASS" if clean_df is not None else "FAIL",
    f"Shape={clean_df.shape}" if clean_df is not None else "Cleaned CSV unavailable"
)

record(
    "Delivery gap dataset verification",
    "PASS" if gap_df is not None else "FAIL",
    f"Shape={gap_df.shape}" if gap_df is not None else "Delivery gap CSV unavailable"
)

record("Project directory verification", "PASS", str(PROJECT_ROOT))

heading("DATA QUALITY AND STRUCTURE VERIFICATION")

required_columns = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "Shipping Mode",
    "Order Region",
    "Market",
    "Customer Segment"
]

if clean_df is not None:
    missing_columns = [
        name for name in required_columns
        if find_exact_column(clean_df, [name]) is None
    ]
    record(
        "Column verification",
        "PASS" if not missing_columns else "FAIL",
        f"Missing columns={missing_columns}; total columns={len(clean_df.columns)}"
    )

    missing_count = int(clean_df.isna().sum().sum())
    record("Missing-value verification", "PASS",
           f"Missing values inspected={missing_count}")

    duplicate_count = int(clean_df.duplicated().sum())
    record(
        "Duplicate verification",
        "PASS" if duplicate_count == 0 else "WARNING",
        f"Exact duplicate rows={duplicate_count}"
    )

    actual_col_clean = find_exact_column(
        clean_df, ["Days for shipping (real)"]
    )
    scheduled_col_clean = find_exact_column(
        clean_df, ["Days for shipment (scheduled)"]
    )
    status_col_clean = find_exact_column(clean_df, ["Delivery Status"])
    risk_col_clean = find_exact_column(clean_df, ["Late_delivery_risk"])
    mode_col_clean = find_exact_column(clean_df, ["Shipping Mode"])
    region_col_clean = find_exact_column(clean_df, ["Order Region"])
    market_col_clean = find_exact_column(clean_df, ["Market"])
    segment_col_clean = find_exact_column(clean_df, ["Customer Segment"])

    actual_clean = (
        pd.to_numeric(clean_df[actual_col_clean], errors="coerce")
        if actual_col_clean else None
    )
    scheduled_clean = (
        pd.to_numeric(clean_df[scheduled_col_clean], errors="coerce")
        if scheduled_col_clean else None
    )

    if actual_clean is not None and scheduled_clean is not None:
        invalid_duration = (
            actual_clean.isna() | scheduled_clean.isna() |
            (actual_clean < 0) | (scheduled_clean < 0)
        )
        record(
            "Shipping duration verification",
            "PASS" if int(invalid_duration.sum()) == 0 else "WARNING",
            f"Invalid duration records={int(invalid_duration.sum())}; "
            f"zero scheduled-duration records={int((scheduled_clean == 0).sum())}"
        )
    else:
        record("Shipping duration verification", "FAIL",
               "Actual or scheduled shipping duration column is missing")

    if risk_col_clean:
        risk_set = set(
            pd.to_numeric(clean_df[risk_col_clean], errors="coerce").dropna().unique()
        )
        record(
            "Late delivery risk verification",
            "PASS" if risk_set.issubset({0, 1}) else "FAIL",
            f"Observed values={sorted(risk_set)}"
        )
    else:
        record("Late delivery risk verification", "FAIL",
               "Late delivery risk column missing")

    record(
        "Regional and market verification",
        "PASS" if region_col_clean and market_col_clean else "FAIL",
        f"Regions={clean_df[region_col_clean].nunique() if region_col_clean else 'Missing'}; "
        f"Markets={clean_df[market_col_clean].nunique() if market_col_clean else 'Missing'}"
    )

    record(
        "Customer segment verification",
        "PASS" if segment_col_clean else "FAIL",
        f"Segments={clean_df[segment_col_clean].nunique() if segment_col_clean else 'Missing'}"
    )
else:
    record("Column verification", "FAIL", "Cleaned dataset unavailable")
    record("Missing-value verification", "FAIL", "Cleaned dataset unavailable")
    record("Duplicate verification", "FAIL", "Cleaned dataset unavailable")
    record("Shipping duration verification", "FAIL", "Cleaned dataset unavailable")
    record("Late delivery risk verification", "FAIL", "Cleaned dataset unavailable")
    record("Regional and market verification", "FAIL", "Cleaned dataset unavailable")
    record("Customer segment verification", "FAIL", "Cleaned dataset unavailable")

heading("INDEPENDENT DELIVERY GAP RECONSTRUCTION")

analysis_df = gap_df if gap_df is not None else clean_df

actual_col = find_exact_column(
    analysis_df, ["Days for shipping (real)"]
) if analysis_df is not None else None

scheduled_col = find_exact_column(
    analysis_df, ["Days for shipment (scheduled)"]
) if analysis_df is not None else None

status_col = find_exact_column(
    analysis_df, ["Delivery Status"]
) if analysis_df is not None else None

mode_col = find_exact_column(
    analysis_df, ["Shipping Mode"]
) if analysis_df is not None else None

region_col = find_exact_column(
    analysis_df, ["Order Region"]
) if analysis_df is not None else None

market_col = find_exact_column(
    analysis_df, ["Market"]
) if analysis_df is not None else None

segment_col = find_exact_column(
    analysis_df, ["Customer Segment"]
) if analysis_df is not None else None

risk_col = find_exact_column(
    analysis_df, ["Late_delivery_risk"]
) if analysis_df is not None else None

calculated_gap = None
eligible = None
actual_values = None
scheduled_values = None

if analysis_df is not None and actual_col and scheduled_col:
    actual_values = pd.to_numeric(
        analysis_df[actual_col], errors="coerce"
    )
    scheduled_values = pd.to_numeric(
        analysis_df[scheduled_col], errors="coerce"
    )

    # Delivery gap is reconstructed from the two source duration fields.
    calculated_gap = actual_values - scheduled_values

    eligible = (
        actual_values.notna() &
        scheduled_values.notna() &
        (actual_values >= 0) &
        (scheduled_values >= 0) &
        calculated_gap.notna()
    )

    if status_col:
        eligible &= ~analysis_df[status_col].astype(str).str.lower().str.contains(
            "cancel", na=False
        )

    # Zero scheduled shipping duration is retained as valid Same Day data.
    gap_valid_count = int(calculated_gap.notna().sum())
    eligible_count = int(eligible.sum())

    # Compare the reconstructed values with an exact numeric gap field if one exists.
    stored_gap_col = find_exact_column(
        analysis_df,
        ["Delivery Gap (Days)", "Delivery Gap Days", "Delivery Gap", "delivery_gap_days"]
    )

    if stored_gap_col:
        stored_gap = pd.to_numeric(analysis_df[stored_gap_col], errors="coerce")
        comparable = stored_gap.notna() & calculated_gap.notna()
        mismatches = int(
            (stored_gap[comparable] != calculated_gap[comparable]).sum()
        )
        gap_detail = (
            f"Reconstructed from actual minus scheduled; "
            f"stored comparison field={stored_gap_col}; "
            f"compared={int(comparable.sum())}; mismatches={mismatches}"
        )
        gap_status = "PASS" if mismatches == 0 else "FAIL"
    else:
        gap_detail = (
            "Reconstructed directly from actual minus scheduled duration; "
            "no exact stored numeric gap column required"
        )
        gap_status = "PASS"

    record(
        "Delivery gap verification",
        gap_status,
        f"{gap_detail}; valid gaps={gap_valid_count}; "
        f"minimum={calculated_gap.min()}; maximum={calculated_gap.max()}"
    )

    delayed_count = int((eligible & (calculated_gap > 0)).sum())
    exact_on_time_count = int((eligible & (calculated_gap == 0)).sum())
    early_count = int((eligible & (calculated_gap < 0)).sum())

    record(
        "Delivery classification verification",
        "PASS",
        f"Delayed={delayed_count}; exact on-time={exact_on_time_count}; "
        f"early={early_count}; eligible={eligible_count}"
    )

    # Save a reconstructed audit dataset, without replacing the original source.
    reconstructed = analysis_df.copy()
    reconstructed["Verification Delivery Gap (Days)"] = calculated_gap
    reconstructed["Verification Delivery Classification"] = np.select(
        [
            calculated_gap > 0,
            calculated_gap == 0,
            calculated_gap < 0
        ],
        ["Delayed", "Exactly On Time", "Early"],
        default="Invalid"
    )
    reconstructed["Verification Eligible Record"] = eligible
    reconstructed.to_csv(
        TABLE_DIR / "final_verification_reconstructed_delivery_gap.csv",
        index=False
    )
else:
    record(
        "Delivery gap verification",
        "FAIL",
        "Actual and scheduled duration fields were not both available"
    )
    record(
        "Delivery classification verification",
        "FAIL",
        "Cannot classify records without reconstructed delivery gaps"
    )

heading("ANALYTICAL CALCULATION VERIFICATION")

mode_summary = pd.DataFrame()
regional_summary = pd.DataFrame()
market_summary = pd.DataFrame()
segment_summary = pd.DataFrame()
kpi_values = {}

if analysis_df is not None and calculated_gap is not None and eligible is not None:
    eligible_count = int(eligible.sum())
    eligible_gap = calculated_gap.loc[eligible]

    delayed_mask = eligible_gap > 0
    delayed_count = int(delayed_mask.sum())

    average_positive_delay = (
        float(eligible_gap.loc[delayed_mask].mean())
        if delayed_count else 0.0
    )

    sla_rate = (
        float((eligible_gap <= 0).mean() * 100)
        if eligible_count else np.nan
    )

    record(
        "Delivery performance calculations",
        "PASS" if eligible_count > 0 else "FAIL",
        f"Eligible={eligible_count}; delayed={delayed_count}; "
        f"average positive delay={average_positive_delay:.4f} days"
    )

    record(
        "SLA calculations",
        "PASS" if eligible_count > 0 else "FAIL",
        f"SLA-compliant records={int((eligible_gap <= 0).sum())}; "
        f"eligible={eligible_count}; SLA compliance={sla_rate:.4f}%"
    )

    if mode_col:
        mode_data = pd.DataFrame({
            "Shipping Mode": analysis_df.loc[eligible, mode_col].astype(str).values,
            "Delivery Gap": eligible_gap.values
        })

        mode_summary = mode_data.groupby("Shipping Mode").agg(
            Records=("Delivery Gap", "count"),
            Average_Gap=("Delivery Gap", "mean"),
            SLA_Compliance=("Delivery Gap", lambda x: (x <= 0).mean() * 100),
            Average_Positive_Delay=(
                "Delivery Gap",
                lambda x: x[x > 0].mean() if (x > 0).any() else 0
            )
        ).reset_index()

        if risk_col:
            risk_subset = pd.DataFrame({
                "Shipping Mode": analysis_df.loc[eligible, mode_col].astype(str).values,
                "Risk": pd.to_numeric(
                    analysis_df.loc[eligible, risk_col], errors="coerce"
                ).values
            })
            risk_by_mode = risk_subset.groupby("Shipping Mode")["Risk"].mean() * 100
            mode_summary["Late_Risk"] = mode_summary["Shipping Mode"].map(risk_by_mode)
        else:
            mode_summary["Late_Risk"] = (
                mode_data.groupby("Shipping Mode")["Delivery Gap"]
                .apply(lambda x: (x > 0).mean() * 100)
                .reindex(mode_summary["Shipping Mode"]).values
            )

        max_mode_delay = mode_summary["Average_Positive_Delay"].max()
        mode_summary["Inverse_Late_Risk"] = 100 - mode_summary["Late_Risk"]

        mode_summary["Relative_Delay_Score"] = np.where(
            max_mode_delay > 0,
            100 * (
                1 - mode_summary["Average_Positive_Delay"] / max_mode_delay
            ),
            100
        )

        mode_summary["Shipping_Mode_Efficiency_Index"] = (
            0.50 * mode_summary["SLA_Compliance"] +
            0.30 * mode_summary["Inverse_Late_Risk"] +
            0.20 * mode_summary["Relative_Delay_Score"]
        ).clip(0, 100)

        mode_summary.to_csv(
            TABLE_DIR / "final_verification_shipping_mode.csv",
            index=False
        )

        record(
            "Shipping mode calculations",
            "PASS" if len(mode_summary) > 0 else "FAIL",
            f"Independently recalculated {len(mode_summary)} shipping modes"
        )
    else:
        record("Shipping mode calculations", "FAIL", "Shipping Mode column missing")

    if region_col:
        regional_data = pd.DataFrame({
            "Region": analysis_df.loc[eligible, region_col].astype(str).values,
            "Delivery Gap": eligible_gap.values
        })

        regional_summary = regional_data.groupby("Region").agg(
            Records=("Delivery Gap", "count"),
            Average_Gap=("Delivery Gap", "mean"),
            Average_Positive_Delay=(
                "Delivery Gap",
                lambda x: x[x > 0].mean() if (x > 0).any() else 0
            ),
            SLA_Compliance=("Delivery Gap", lambda x: (x <= 0).mean() * 100)
        ).reset_index()

        max_region_delay = regional_summary["Average_Positive_Delay"].max()
        regional_summary["Regional_Delay_Index"] = np.where(
            max_region_delay > 0,
            regional_summary["Average_Positive_Delay"] / max_region_delay * 100,
            0
        )

        regional_summary.to_csv(
            TABLE_DIR / "final_verification_regional.csv",
            index=False
        )

        record(
            "Regional analysis calculations",
            "PASS" if len(regional_summary) > 0 else "FAIL",
            f"Independently recalculated {len(regional_summary)} regions"
        )
    else:
        record("Regional analysis calculations", "FAIL", "Order Region column missing")

    if market_col:
        market_data = pd.DataFrame({
            "Market": analysis_df.loc[eligible, market_col].astype(str).values,
            "Delivery Gap": eligible_gap.values
        })

        market_summary = market_data.groupby("Market").agg(
            Records=("Delivery Gap", "count"),
            Average_Gap=("Delivery Gap", "mean"),
            SLA_Compliance=("Delivery Gap", lambda x: (x <= 0).mean() * 100)
        ).reset_index()

        market_summary.to_csv(
            TABLE_DIR / "final_verification_market.csv",
            index=False
        )

        record(
            "Market analysis calculations",
            "PASS" if len(market_summary) > 0 else "FAIL",
            f"Independently recalculated {len(market_summary)} markets"
        )
    else:
        record("Market analysis calculations", "FAIL", "Market column missing")

    if segment_col:
        segment_data = pd.DataFrame({
            "Customer Segment": analysis_df.loc[eligible, segment_col].astype(str).values,
            "Delivery Gap": eligible_gap.values
        })

        segment_summary = segment_data.groupby("Customer Segment").agg(
            Records=("Delivery Gap", "count"),
            Average_Gap=("Delivery Gap", "mean"),
            SLA_Compliance=("Delivery Gap", lambda x: (x <= 0).mean() * 100),
            Late_Rate=("Delivery Gap", lambda x: (x > 0).mean() * 100)
        ).reset_index()

        segment_summary.to_csv(
            TABLE_DIR / "final_verification_customer_segment.csv",
            index=False
        )

        record(
            "Customer segment calculations",
            "PASS" if len(segment_summary) > 0 else "FAIL",
            f"Independently recalculated {len(segment_summary)} segments"
        )
    else:
        record("Customer segment calculations", "FAIL", "Customer Segment column missing")

    if risk_col:
        risk_values = pd.to_numeric(
            analysis_df.loc[eligible, risk_col], errors="coerce"
        ).dropna()

        risk_ratio = (
            float(risk_values.mean() * 100)
            if len(risk_values) else np.nan
        )

        record(
            "Late delivery risk calculations",
            "PASS" if len(risk_values) > 0 else "FAIL",
            f"Valid risk records={len(risk_values)}; risk ratio={risk_ratio:.4f}%"
        )
    else:
        risk_ratio = np.nan
        record("Late delivery risk calculations", "FAIL",
               "Late_delivery_risk column missing")

    kpi_values = {
        "On-Time Delivery Rate (%)": sla_rate,
        "Average Delivery Delay (Days)": average_positive_delay,
        "Late Delivery Risk Ratio": risk_ratio,
        "Shipping Mode Efficiency Index": (
            float(mode_summary["Shipping_Mode_Efficiency_Index"].mean())
            if not mode_summary.empty else np.nan
        ),
        "Regional Delay Index": (
            float(regional_summary["Regional_Delay_Index"].mean())
            if not regional_summary.empty else np.nan
        )
    }

    valid_kpis = sum(
        value is not None and np.isfinite(value)
        for value in kpi_values.values()
    )

    record(
        "Five KPI calculations",
        "PASS" if valid_kpis == 5 else "FAIL",
        f"Valid KPI calculations={valid_kpis}/5; values={kpi_values}"
    )

    pd.DataFrame([
        {"KPI": name, "Value": value}
        for name, value in kpi_values.items()
    ]).to_csv(
        TABLE_DIR / "final_verification_five_kpis.csv",
        index=False
    )
else:
    for name in [
        "Delivery performance calculations",
        "Shipping mode calculations",
        "Regional analysis calculations",
        "Market analysis calculations",
        "Customer segment calculations",
        "SLA calculations",
        "Late delivery risk calculations",
        "Five KPI calculations"
    ]:
        record(name, "FAIL", "Independent gap reconstruction unavailable")

heading("CHART DATA VERIFICATION")

chart_files = find_files(
    [PROJECT_ROOT / "outputs", PROJECT_ROOT / "dashboard"],
    ["*.csv"]
)

chart_candidates = [
    p for p in chart_files
    if any(term in p.name.lower() for term in [
        "chart", "plot", "figure", "trend", "performance",
        "shipping_mode", "regional", "market", "segment", "kpi"
    ])
]

readable_chart_files = sum(read_csv(p) is not None for p in chart_candidates)

record(
    "All chart data verification",
    "PASS" if readable_chart_files > 0 else "WARNING",
    f"Chart-related CSV files={len(chart_candidates)}; readable={readable_chart_files}"
)

heading("STREAMLIT DASHBOARD VERIFICATION")

if APP_FILE.exists():
    app_source = APP_FILE.read_text(encoding="utf-8", errors="ignore")
    app_lower = app_source.lower()

    try:
        ast.parse(app_source)
        record("Dashboard Python syntax", "PASS", "Python syntax valid")
    except SyntaxError as error:
        record("Dashboard Python syntax", "FAIL", error)

    module_checks = {
        "Delivery Performance module": [
            "delivery performance overview", "delivery performance"
        ],
        "Delay Risk module": [
            "delay risk analysis", "delay risk"
        ],
        "Shipping Mode module": [
            "shipping mode comparison", "shipping mode"
        ],
        "Regional and Market module": [
            "regional and market heatmaps", "regional", "market"
        ]
    }

    for name, terms in module_checks.items():
        found = any(term in app_lower for term in terms)
        record(name, "PASS" if found else "WARNING",
               "Module reference found" if found else "Module reference not found")

    for name, terms in {
        "Shipping mode filter": ["shipping mode"],
        "Region and market selector": ["order region", "market"],
        "Customer segment filter": ["customer segment"]
    }.items():
        found = all(term in app_lower for term in terms)
        record(name, "PASS" if found else "WARNING",
               "Filter reference found" if found else "Filter reference missing")

    genuine_date_col = find_exact_column(
        clean_df,
        [
            "Order Date", "Shipping Date", "Delivery Date",
            "Actual Delivery Date", "Scheduled Delivery Date"
        ]
    ) if clean_df is not None else None

    if genuine_date_col:
        date_selector_found = "date_input" in app_lower
        record(
            "Date range selector",
            "PASS" if date_selector_found else "WARNING",
            f"Genuine date column={genuine_date_col}"
        )
    else:
        record(
            "Date range selector",
            "PASS",
            "Not applicable: no genuine operational date column; synthetic dates not created"
        )

    download_found = any(
        term in app_lower
        for term in ["download_button", "csv_download", "download"]
    )
    record(
        "Report downloads",
        "PASS" if download_found else "WARNING",
        "Download implementation detected" if download_found else "Download implementation not detected"
    )

    smoke_files = find_files(
        [PROJECT_ROOT / "outputs", PROJECT_ROOT / "deployment"],
        ["*smoke*test*.json", "*dashboard*test*.json"]
    )

    record(
        "Dashboard performance",
        "PASS" if smoke_files else "WARNING",
        f"Saved smoke test evidence files={len(smoke_files)}"
    )

    error_handling = "try:" in app_source and "except" in app_source
    record(
        "Application error handling",
        "PASS" if error_handling else "WARNING",
        "Exception handling detected" if error_handling else "Explicit exception handling not detected"
    )
else:
    for name in [
        "Dashboard Python syntax", "Delivery Performance module",
        "Delay Risk module", "Shipping Mode module",
        "Regional and Market module", "Shipping mode filter",
        "Region and market selector", "Customer segment filter",
        "Date range selector", "Report downloads",
        "Dashboard performance", "Application error handling"
    ]:
        record(name, "FAIL", "app.py unavailable")

heading("RESEARCH PAPER AND EXECUTIVE SUMMARY VERIFICATION")

research_files = find_files(
    [
        PROJECT_ROOT / "research_paper",
        PROJECT_ROOT / "documentation",
        PROJECT_ROOT / "outputs"
    ],
    [
        "*research*paper*.docx", "*research*paper*.pdf",
        "*forecasting*research*.docx", "*forecasting*research*.pdf"
    ]
)

executive_files = find_files(
    [
        PROJECT_ROOT / "government_executive_summary",
        PROJECT_ROOT / "executive_summary",
        PROJECT_ROOT / "documentation",
        PROJECT_ROOT / "outputs"
    ],
    ["*executive*summary*.docx", "*executive*summary*.pdf"]
)

def verify_document_group(label, files, required_terms):
    valid = [p for p in files if valid_document(p)]

    if not valid:
        record(label, "FAIL", "No readable document found")
        return

    record(label, "PASS", f"Readable documents={len(valid)}")

    docx_files = [p for p in valid if p.suffix.lower() == ".docx"]
    if docx_files:
        content = " ".join(extract_docx_text(p) for p in docx_files)
        missing = [
            term for term in required_terms
            if normalize(term) not in normalize(content)
        ]
        record(
            label + " content verification",
            "PASS" if not missing else "WARNING",
            f"Missing required terms={missing}"
        )

verify_document_group(
    "Research paper verification",
    research_files,
    ["Abstract", "Introduction", "Methodology", "Conclusion"]
)

verify_document_group(
    "Executive summary verification",
    executive_files,
    ["Executive Overview", "Data Limitations", "Conclusion"]
)

heading("PROJECT DOCUMENTATION GENERATION")

documentation_content = {
    "README.md": """
# Analysis in Global Supply Chain Operations

## Project overview

This project analyzes delivery performance, delivery delays, shipping risk,
shipping mode efficiency, regional differences, market performance, and
customer segment exposure using the supplied logistics dataset.

## Main analytical outputs

- Cleaned logistics dataset
- Delivery gap calculations and classifications
- Overall delivery performance analysis
- Shipping mode efficiency analysis
- Regional and market analysis
- Customer segment analysis
- Five required logistics KPIs
- Streamlit dashboard
- Research paper
- Government executive summary

## Dataset

Raw source: `data/raw/APL_Logistics.xlsx`

Cleaned dataset: `data/processed/cleaned_logistics_data.csv`

Delivery gap dataset: `data/processed/delivery_gap_analysis.csv`

The current source contains 180,519 observations. The cleaned dataset contains
32 columns. The delivery gap analysis dataset contains 39 columns.

## Delivery gap definition

Delivery gap = actual shipping duration - scheduled shipping duration.

- Positive gap: delayed delivery
- Zero gap: exactly on-time delivery
- Negative gap: early delivery

SLA compliance is defined as delivery gap less than or equal to zero.
Cancelled shipments and invalid duration records are excluded from the eligible
delivery performance population. Valid zero scheduled-duration Same Day records
are retained.

## Required KPIs

1. On-Time Delivery Rate
2. Average Delivery Delay
3. Late Delivery Risk Ratio
4. Shipping Mode Efficiency Index
5. Regional Delay Index

The Shipping Mode Efficiency Index and Regional Delay Index are project-defined
analytical indices, not external industry standards.

## Dashboard

Run the dashboard from the project root:

```powershell
streamlit run app.py
```

The dashboard contains four modules:

- Delivery Performance Overview
- Delay Risk Analysis
- Shipping Mode Comparison
- Regional and Market Heatmaps

## Data limitations

The supplied dataset does not contain a genuine operational date field suitable
for a date-range selector. Synthetic dates must not be generated. Customer
coordinates must not be interpreted as verified shipment destination
coordinates. Customer IDs must not be treated as order IDs.

## Main folders

- `data`: source and processed datasets
- `notebooks`: analytical notebooks
- `src`: reusable analytical modules
- `dashboard`: dashboard-related resources
- `models`: model artifacts, if applicable
- `outputs`: tables, figures, reports, and forecasts
- `research_paper`: research paper deliverables
- `government_executive_summary`: executive summary deliverables
- `documentation`: project documentation
""",

    "INSTALLATION_GUIDE.md": """
# Installation Guide

## Requirements

- Windows
- Python 3.10 or later
- pip
- Access to the project directory

## Open the project

Open PowerShell and run:

```powershell
cd "C:\\Users\\hp\\Analysis in Global Supply Chain Operations"
```

## Create a virtual environment

```powershell
python -m venv .venv
```

## Activate the environment

```powershell
.venv\\Scripts\\Activate.ps1
```

If PowerShell blocks activation, use the appropriate execution policy permitted
by your computer administrator, or run the environment Python directly.

## Install dependencies

```powershell
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

## Launch the dashboard

```powershell
streamlit run app.py
```

## Run notebooks

Start Jupyter from the project root so project-relative paths resolve correctly:

```powershell
jupyter notebook
```

## Troubleshooting

- Confirm that the project root contains `app.py`.
- Confirm that the raw workbook is in `data/raw`.
- Confirm that the processed CSV files are in `data/processed`.
- Check that the active Python environment has the packages in requirements.txt.
- Do not fabricate an operational date column when the source data does not
  contain one.
""",

    "USER_GUIDE.md": """
# User Guide

## Dashboard access

From the project root, launch:

```powershell
streamlit run app.py
```

Open the local URL displayed by Streamlit.

## Dashboard modules

### Delivery Performance Overview

Use this module to review delivery performance, eligible shipment population,
delivery gap distribution, and SLA compliance.

### Delay Risk Analysis

Use this module to examine the supplied late-delivery risk indicator and
delivery delay patterns. The existing risk field is a dataset label and should
not be described as a newly trained prediction.

### Shipping Mode Comparison

Compare Standard Class, Second Class, First Class, and Same Day using the
available shipping mode performance measures.

### Regional and Market Heatmaps

Compare regional and market delivery performance using categorical
aggregations. Customer latitude and longitude are not verified shipment
destination coordinates and should not be used to create shipment destination
maps.

## Filters and downloads

Use the available shipping mode, region, market, and customer segment filters.
Use report download controls where provided.

## KPI interpretation

- On-Time Delivery Rate: share of eligible records with delivery gap <= 0.
- Average Delivery Delay: average positive delivery gap among delayed records.
- Late Delivery Risk Ratio: average valid binary late-risk label multiplied by 100.
- Shipping Mode Efficiency Index: project-defined composite index.
- Regional Delay Index: regional average positive delay relative to the highest
  regional average positive delay.

## Date filter limitation

The supplied dataset has no genuine operational date field. A date-range
selector is therefore unavailable until a valid date field is supplied.

## Interpretation cautions

Cancelled shipments and invalid duration records are excluded from delivery
performance denominators. Valid zero scheduled-duration Same Day records are
retained. Corporate is a customer segment and is not automatically equivalent
to premium or enterprise customers.
""",

    "TECHNICAL_DOCUMENTATION.md": """
# Technical Documentation

## Project structure

The project separates raw data, processed data, notebooks, reusable source
modules, dashboard code, reports, figures, and final submission artifacts.

## Data pipeline

1. Load the original logistics workbook.
2. Inspect and validate source columns and values.
3. Clean the dataset while preserving valid observations.
4. Calculate delivery gap as actual duration minus scheduled duration.
5. Classify delivery records as delayed, exactly on time, or early.
6. Calculate delivery performance by mode, region, market, and segment.
7. Calculate the five required KPIs.
8. Provide analytical outputs through the Streamlit dashboard and reports.

## Eligibility rules

A delivery record is eligible for delivery performance calculations when
actual and scheduled shipping durations are present and non-negative and the
shipment is not cancelled.

A scheduled duration of zero is valid and represents Same Day shipping.
Negative delivery gaps are early deliveries and are SLA compliant.

## KPI formulas

On-Time Delivery Rate:

SLA-compliant eligible records / eligible records * 100

SLA compliance means delivery gap <= 0.

Average Delivery Delay:

Mean of positive delivery gaps among delayed eligible records.

Late Delivery Risk Ratio:

Mean of valid binary Late_delivery_risk values * 100.

Shipping Mode Efficiency Index:

50% SLA compliance + 30% inverse late risk + 20% relative delay-gap score.

Regional Delay Index:

Regional average positive delay / maximum regional average positive delay * 100.

The two composite indices are project-defined metrics.

## Dashboard architecture

The Streamlit application provides four analytical modules:
delivery performance overview, delay risk analysis, shipping mode comparison,
and regional and market heatmaps.

## Data limitations

The source data does not contain a genuine operational date field. The project
does not create synthetic dates. Customer coordinates are not verified
shipment destination coordinates. Customer IDs are not order IDs. The
Late_delivery_risk field is an existing binary label.

## Verification

The final verification process checks source files, schema, data quality,
duration and gap calculations, analytical groupings, KPIs, dashboard source,
documentation, output files, and ZIP archive integrity.
"""
}

for filename, content in documentation_content.items():
    write_text(PROJECT_ROOT / filename, content)

record("README verification",
       "PASS" if (PROJECT_ROOT / "README.md").exists() else "FAIL",
       "README.md generated and saved")

record("Installation guide verification",
       "PASS" if (PROJECT_ROOT / "INSTALLATION_GUIDE.md").exists() else "FAIL",
       "INSTALLATION_GUIDE.md generated and saved")

record("User guide verification",
       "PASS" if (PROJECT_ROOT / "USER_GUIDE.md").exists() else "FAIL",
       "USER_GUIDE.md generated and saved")

record("Technical documentation verification",
       "PASS" if (PROJECT_ROOT / "TECHNICAL_DOCUMENTATION.md").exists() else "FAIL",
       "TECHNICAL_DOCUMENTATION.md generated and saved")

heading("REQUIREMENTS AND DIRECTORY VERIFICATION")

requirements_file = PROJECT_ROOT / "requirements.txt"

if requirements_file.exists() and requirements_file.stat().st_size > 0:
    record("Requirements file verification", "PASS",
           f"Found {requirements_file.name}")
else:
    record("Requirements file verification", "FAIL",
           "requirements.txt is missing or empty")

required_directories = [
    "data/raw", "data/processed", "data/interim",
    "dashboard", "dashboard/pages", "documentation",
    "models", "notebooks", "outputs/figures",
    "outputs/forecasts", "outputs/reports",
    "outputs/tables", "research_paper", "src", "tests"
]

for directory in required_directories:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

missing_directories = [
    d for d in required_directories if not (PROJECT_ROOT / d).exists()
]

record(
    "Project directory structure verification",
    "PASS" if not missing_directories else "FAIL",
    f"Missing directories={missing_directories}"
)

project_files = [
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file()
    and not any(part in {
        ".git", "__pycache__", ".ipynb_checkpoints",
        ".venv", ".venv_deployment", ".deployment_test_env",
        "venv", "env", "node_modules"
    } for part in p.parts)
]

output_files = [p for p in project_files if "outputs" in p.parts]

record(
    "Output file verification",
    "PASS" if len(output_files) > 0 else "WARNING",
    f"Project files={len(project_files)}; output files={len(output_files)}"
)

heading("FINAL VALIDATION REPORT AND COMPLETION REPORT")

report_csv = REPORT_DIR / "final_project_validation_report.csv"
report_json = REPORT_DIR / "final_project_validation_report.json"
completion_file = REPORT_DIR / "final_project_completion_report.txt"

archive_path = ARCHIVE_DIR / (
    "final_submission_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".zip"
)

def make_summary():
    frame = pd.DataFrame(results)
    return {
        "project": "Analysis in Global Supply Chain Operations",
        "project_directory": str(PROJECT_ROOT),
        "verification_timestamp": datetime.now().isoformat(timespec="seconds"),
        "total_checks": len(frame),
        "passed": int((frame["Status"] == "PASS").sum()),
        "failed": int((frame["Status"] == "FAIL").sum()),
        "warnings": int((frame["Status"] == "WARNING").sum()),
        "checks": results
    }

def save_final_reports(archive_entries=None, archive_integrity=None):
    summary = make_summary()
    summary["archive_path"] = str(archive_path)
    summary["archive_entries"] = archive_entries
    summary["archive_integrity"] = archive_integrity

    summary["overall_status"] = (
        "ALL CHECKS PASSED"
        if summary["failed"] == 0 and summary["warnings"] == 0 and archive_integrity
        else "PASS WITH WARNINGS"
        if summary["failed"] == 0 and archive_integrity
        else "FAILED"
    )

    pd.DataFrame(results).to_csv(report_csv, index=False)
    report_json.write_text(
        json.dumps(summary, indent=4, default=str),
        encoding="utf-8"
    )

    completion = f"""
FINAL PROJECT COMPLETION REPORT
================================

Project: Analysis in Global Supply Chain Operations
Project directory: {PROJECT_ROOT}
Verification timestamp: {summary['verification_timestamp']}

VALIDATION SUMMARY
------------------
Total checks: {summary['total_checks']}
Passed: {summary['passed']}
Failed: {summary['failed']}
Warnings: {summary['warnings']}
Overall status: {summary['overall_status']}

DATASET SUMMARY
---------------
Raw dataset shape: {raw_df.shape if raw_df is not None else 'Unavailable'}
Cleaned dataset shape: {clean_df.shape if clean_df is not None else 'Unavailable'}
Delivery gap dataset shape: {gap_df.shape if gap_df is not None else 'Unavailable'}

DELIVERY GAP DEFINITION
-----------------------
Delivery gap was independently reconstructed as:
Actual shipping duration minus scheduled shipping duration.

Cancelled and invalid-duration records are excluded from the eligible population.
Valid zero scheduled-duration Same Day records are retained.
Positive gaps are delayed, zero gaps are exactly on time, and negative gaps are early.

DATE LIMITATION
--------------
No genuine operational date field was identified.
The dashboard date-range selector remains unavailable until genuine date data is supplied.
No synthetic dates were generated.

ARCHIVE
-------
Archive: {archive_path}
Entries: {archive_entries}
Integrity: {archive_integrity}

REPORT FILES
------------
{report_csv}
{report_json}
{completion_file}
"""
    completion_file.write_text(completion.strip() + "\n", encoding="utf-8")
    return summary

record(
    "Final validation report",
    "PASS",
    "Final CSV, JSON and completion report are generated"
)

heading("SUBMISSION ARCHIVE GENERATION")

excluded_directories = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    ".venv", ".venv_deployment", ".deployment_test_env",
    "venv", "env", "node_modules"
}

def build_archive():
    count = 0
    with zipfile.ZipFile(
        archive_path, "w", compression=zipfile.ZIP_DEFLATED
    ) as archive:
        for path in PROJECT_ROOT.rglob("*"):
            if not path.is_file():
                continue
            if path == archive_path or ARCHIVE_DIR in path.parents:
                continue
            if any(part in excluded_directories for part in path.parts):
                continue
            if path.suffix.lower() in {".pyc", ".pyo"}:
                continue
            try:
                archive.write(
                    path,
                    arcname=str(path.relative_to(PROJECT_ROOT))
                )
                count += 1
            except Exception:
                pass
    return count

archive_entries = build_archive()

record(
    "Submission archive generation",
    "PASS" if archive_path.exists() and archive_path.stat().st_size > 0 else "FAIL",
    f"Archive={archive_path}; entries={archive_entries}; "
    f"size={archive_path.stat().st_size if archive_path.exists() else 0} bytes"
)

heading("ZIP ARCHIVE INTEGRITY VERIFICATION")

try:
    with zipfile.ZipFile(archive_path, "r") as archive:
        bad_file = archive.testzip()
        archive_entries = len(archive.namelist())
    archive_integrity = bad_file is None and archive_entries > 0
    integrity_detail = f"Bad file={bad_file}; entries={archive_entries}"
except Exception as error:
    archive_integrity = False
    integrity_detail = str(error)

record(
    "ZIP archive integrity test",
    "PASS" if archive_integrity else "FAIL",
    integrity_detail
)

# Save reports with the completed archive verification result.
summary = save_final_reports(archive_entries, archive_integrity)

# Rebuild archive so it includes the final validation reports.
archive_entries = build_archive()

with zipfile.ZipFile(archive_path, "r") as archive:
    final_bad_file = archive.testzip()
    final_archive_entries = len(archive.namelist())

final_archive_integrity = (
    final_bad_file is None and final_archive_entries > 0
)

# Refresh the final reports after the final archive test.
summary = save_final_reports(
    final_archive_entries,
    final_archive_integrity
)

# Include the refreshed final reports in the archive.
archive_entries = build_archive()

with zipfile.ZipFile(archive_path, "r") as archive:
    last_bad_file = archive.testzip()
    last_archive_entries = len(archive.namelist())

last_archive_integrity = (
    last_bad_file is None and last_archive_entries > 0
)

heading("FINAL PROJECT VERIFICATION RESULTS")

final_frame = pd.DataFrame(results)
print(final_frame.to_string(index=False))

passed = int((final_frame["Status"] == "PASS").sum())
failed = int((final_frame["Status"] == "FAIL").sum())
warnings_count = int((final_frame["Status"] == "WARNING").sum())

print("\n" + "=" * 100)
print("TOTAL VALIDATION CHECKS :", len(final_frame))
print("PASSED                  :", passed)
print("FAILED                  :", failed)
print("WARNINGS                :", warnings_count)
print("FINAL ZIP ENTRIES       :", last_archive_entries)
print("FINAL ZIP INTEGRITY     :", "PASS" if last_archive_integrity else "FAIL")
print("FINAL ARCHIVE           :", archive_path)
print("VALIDATION CSV          :", report_csv)
print("VALIDATION JSON         :", report_json)
print("COMPLETION REPORT       :", completion_file)

if failed > 0 or not last_archive_integrity:
    print("\nFINAL PROJECT STATUS: FAILED - review failed checks.")
elif warnings_count > 0:
    print("\nFINAL PROJECT STATUS: PASS WITH WARNINGS - review warnings.")
else:
    print("\nFINAL PROJECT STATUS: ALL CHECKS PASSED")



PROJECT DIRECTORY AND DATASET LOADING
